# Домашняя работа 5. Скользящий контроль изнутри и VC-размерность

**Курс «Машинное обучение», 4 курс, каф. ФН1**

| | |
|---|---|
| К семинару | занятие 5 — Переобучение, смещение–разброс и скользящий контроль |
| Опора | материал семинара 5 и лекций до него |
| Ожидаемое время | 3–4 часа |

На занятии мы пользовались `cross_val_score` и `GridSearchCV` как чёрными ящиками. Дома напишем контроль сами, воспроизведём численный пример из конспекта, измерим смещение «лучшего значения по сетке» и проверим перебором, чему равна VC-размерность линейного классификатора.

## Что нужно сдать

Заполненный ноутбук, в котором:

1. выполнены все ячейки с `# TODO`, код исполняется сверху вниз без ошибок
   в свежем ядре (Kernel → Restart & Run All);
2. под каждым заданием заполнена ячейка **Вывод** — своими словами;
3. графики подписаны: заголовок, оси, легенда;
4. вариант ваш собственный (ячейка ниже).

> Списывание видно сразу: у каждого студента свой датасет и свой набор методов.

In [ ]:
# Служебная ячейка: импорты, стиль графиков, воспроизводимость.
import sys, pathlib, warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

# Модули практикума (variants.py, labdata.py) ищем рядом с ноутбуком.
# Если их нет -- значит, ноутбук открыт в Colab: скачиваем из репозитория курса.
COURSE_FILES_URL = "https://raw.githubusercontent.com/sharipovaka/mltest1/main/notebooks"


def _course_modules_dir():
    here = pathlib.Path.cwd()
    for parent in [here, *here.parents][:4]:
        if (parent / "variants.py").exists():
            return str(parent)
    import urllib.request
    for name in ("variants.py", "labdata.py"):
        if not pathlib.Path(name).exists():
            urllib.request.urlretrieve(f"{COURSE_FILES_URL}/{name}", name)
            print(f"загружен {name} из репозитория курса")
    return str(here)


sys.path.insert(0, _course_modules_dir())
from variants import get_variant, describe_variant  # noqa: E402

RANDOM_STATE = 42          # единый seed на всю работу: результаты воспроизводимы
rng = np.random.default_rng(RANDOM_STATE)

plt.rcParams.update({
    "figure.figsize": (7.5, 4.5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

print("numpy", np.__version__, "| pandas", pd.__version__)

from scipy import optimize
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, KFold, StratifiedKFold
from sklearn.metrics import roc_auc_score

## Индивидуальный вариант

Впишите своё ФИО (или почту) в переменную `STUDENT` — вариант вычисляется детерминированно, при повторном запуске он тот же самый.

In [ ]:
STUDENT = "Фамилия Имя Отчество"   # <-- впишите себя

variant = get_variant(STUDENT, lab=5)
describe_variant(variant)

---
# Задача 1. Скользящий контроль своими руками

Начнём с точного воспроизведения примера 4.9 из конспекта: $y = (2,4,6,8)$,
модель — константа, метод А предсказывает выборочное среднее, метод Б всегда
выдаёт $0$. Лекция даёт $\mathrm{LOO}(A)\approx8.89$ и $\mathrm{LOO}(Б)=30$.

In [ ]:
def loo(method, y):
    """Контроль по отдельным объектам: L обучений на L-1 объекте.

    method -- функция, которая по оставшимся ответам выдаёт прогноз.
    Возвращает (среднюю ошибку, список ошибок по объектам).
    """
    raise NotImplementedError


y_toy = np.array([2.0, 4.0, 6.0, 8.0])
# TODO: воспроизведите таблицу примера 4.9 и оба значения LOO
#       (метод А -- np.mean, метод Б -- всегда 0).

### Задание 1.2. Своя реализация $q$-кратного контроля

Реализуйте `my_kfold(n, q, shuffle, seed)`, возвращающий список пар индексов
`(train_idx, test_idx)`, и `my_cross_val_score` поверх него. Сверьте разбиения
со `sklearn.model_selection.KFold` при `shuffle=False` — они обязаны совпасть.

In [ ]:
def my_kfold(n, q, shuffle=True, seed=0):
    """q непересекающихся блоков; возвращает пары (train_idx, test_idx).

    Подсказка: np.array_split делит массив индексов на почти равные части.
    """
    raise NotImplementedError


def my_cross_val_score(make_estimator, X, y, q=5, seed=0):
    """Средняя квадратичная ошибка по блокам."""
    raise NotImplementedError


# TODO: сверьте своё разбиение со sklearn KFold при shuffle=False.

---
# Задача 2. Вложенный контроль

Если по скользящему контролю **выбран** гиперпараметр, то само значение
$\min_\lambda\mathrm{CV}(\mu_\lambda)$ — уже не честная оценка риска.
Лечение — вложенный контроль: внешний цикл оценивает качество, внутренний
подбирает гиперпараметр, и внутренний цикл не видит внешнего блока.

Измерим смещение на **чистом шуме**, где истинное AUC заведомо равно 0.5.

In [ ]:
N_TRIALS, n_o, n_f = 30, 80, 100
grid = {"C": np.logspace(-4, 2, 10)}
inner = StratifiedKFold(4, shuffle=True, random_state=1)
outer = StratifiedKFold(5, shuffle=True, random_state=2)

# TODO: N_TRIALS раз сгенерируйте ЧИСТЫЙ ШУМ (X случайный, y случайный)
#       и посчитайте две оценки:
#   naive  -- GridSearchCV по сетке C с внешним контролем, взять best_score_;
#   nested -- внешний контроль для оценки, а внутри каждого блока свой
#             GridSearchCV с inner для подбора C.
naive_arr, nested_arr = ..., ...

In [ ]:
from scipy import stats

# TODO: 1) сравните средние обеих оценок с истиной 0.5 (со стандартной ошибкой);
#       2) посчитайте ПАРНУЮ разность (наивно - вложенно), её стандартную ошибку
#          и парный критерий stats.ttest_rel;
#       3) постройте boxplot обеих оценок с линией 0.5.

> **Вывод.** На данных без всякой связи истинное AUC равно 0.5. Что показали обе оценки? Почему парная разность разрешается статистически, а абсолютные значения — нет?
>
> *(ваш ответ здесь)*

---
# Задача 3. VC-размерность перебором

Определение 4.10: $\mathrm{VCdim}(A)$ — наибольшее $d$, для которого существует
набор из $d$ точек, разбиваемый моделью $A$, то есть реализуются **все** $2^d$
разметок. Пример 4.11: для линейных классификаторов в $\mathbb{R}^2$ она равна 3.

Проверим перебором. Линейная разделимость набора $(X,y)$ — это разрешимость
системы $y_i(w^{\mathsf T}x_i + b)\ge1$, то есть задача линейного
программирования с нулевой целевой функцией: важна только совместность.

In [ ]:
from itertools import product


def linearly_separable(X, y):
    """Существуют ли w, b с y_i (w^T x_i + b) >= 1?

    Подсказка: задача ЛП с нулевой целевой функцией. Ограничения приводятся
    к виду A_ub @ z <= b_ub, где z = (w, b). Совместность -> res.status == 0.
    """
    raise NotImplementedError


def shatters(X):
    """Реализуются ли все 2^d разметок? Верните (да/нет, нереализуемая разметка)."""
    raise NotImplementedError


# TODO: проверьте, что 3 точки общего положения разбиваются, а 4 -- нет
#       (и для квадрата, и для случая «точка внутри треугольника»);
#       выведите конкретную нереализуемую разметку в каждом случае.

In [ ]:
# TODO: для n = 2, 3, 5 и d = 2..8 посчитайте долю реализуемых разметок
#       (усредните по 15 случайным наборам точек), сведите в таблицу
#       и постройте график с вертикальными отметками d = n + 1.

> **Вывод.** Какая разметка четырёх точек оказалась нереализуемой в каждом случае? Где ломается кривая доли разделимых разметок и как это связано с $\mathrm{VCdim} = n+1$?
>
> *(ваш ответ здесь)*

## Итоги домашней работы

Кратко ответьте на вопросы:

1. Почему по обучающей ошибке нельзя выбрать сложность модели? Сошлитесь на вложенность моделей.
2. Вам нужно сравнить две модели и заявить, что одна лучше. Опишите протокол в четырёх пунктах так, чтобы в нём не было ни одной из разобранных ловушек.

---

Проверьте перед сдачей: Kernel → Restart & Run All проходит без ошибок,
все ячейки **Вывод** заполнены, графики подписаны.